# 第9章 扩散模型
## Diffusion Models — 去噪扩散概率模型 (DDPM)

**来源：李宏毅《深度学习教程》第9章 | 对应原书第173-176页**

---

## 一、知识地图：全章结构与脉络

```
第9章 扩散模型
├── 扩散模型概述
│   ├── 物理热力学扩散思想
│   ├── 去噪扩散概率模型 (DDPM)
│   └── 代表系统：DALL-E、Imagen、Stable Diffusion
├── 扩散模型的运作原理
│   ├── 生成过程（逆过程/反向过程）
│   │   ├── 从纯噪声开始
│   │   ├── 逐步去噪（如1000步）
│   │   └── 同一个去噪模块反复使用
│   └── 去噪模块的内部结构
│       ├── 噪声预测器 (Noise Predictor)
│       ├── 输入：噪声图片 + 步数编号t
│       └── 输出：预测的噪声（逐像素）
├── 扩散模型的训练
│   ├── 前向过程（扩散过程）
│   │   ├── 从真实图片逐步加噪声
│   │   └── 创造训练数据对
│   └── 训练噪声预测器
│       ├── 输入：第t步的噪声图 + 步数t
│       └── 目标：预测加入的噪声
├── 文生图（Text-to-Image）
│   ├── 训练数据：LAION（58.5亿张图文对）
│   └── 在去噪模块中加入文字条件
└── 为什么预测噪声而非直接预测干净图片？
```

## 二、什么是扩散模型？——核心直觉

### 2.1 米开朗基罗的哲学

扩散模型（Diffusion Model）是一种运用物理热力学扩散思想的生成模型。最著名的版本是**去噪扩散概率模型（DDPM）**。

如今成功的图像生成系统——**DALL-E、Google Imagen、Stable Diffusion、Midjourney**——基本上都采用类似的方法。

> **米开朗基罗说**："塑像就在石头里，我只是把不需要的部分去掉。" 扩散模型做的事情完全相同——它从一张**纯噪声图片**开始，逐步"去掉噪声"，最终显现出清晰的图片。

### 2.2 生成过程的步骤（逆过程/反向过程）

扩散模型生成一张图片的过程称为**逆过程（Reverse Process）**：

1. 从标准高斯分布 $\mathcal{N}(0, I)$ 中采样一个高维向量，维度与目标图片相同（如 $256 \times 256$）
2. 将向量排列成一张"全是噪声"的图片
3. 送入**去噪模块（Denoising Module）**，输出"稍微不那么噪声"的图片
4. 重复第3步，噪声逐步减少，猫的轮廓逐步显现
5. 经过足够多次去噪（通常1000步），得到清晰的图片

去噪次数是预先设定的——开始去噪时步数编号大（如1000=最噪声），接近完成时编号小（如1=最干净）。

### 2.3 去噪模块的设计

同一个去噪模块被反复使用。但每步的输入图片差异极大（从纯噪声到接近完整）。为了让同一模块应对不同阶段，除了输入噪声图片外，还会输入一个**步数编号 $t$**，告知模块当前噪声的严重程度。

> **类比**：就像雕塑家用同一把刻刀，但在粗雕和精雕阶段采用不同的力度。步数编号告诉去噪模块现在是"大块去除"还是"精细加工"。

## 三、去噪模块的内部结构——噪声预测器

### 3.1 核心原理：预测噪声，而非干净图片

去噪模块内部包含一个**噪声预测器（Noise Predictor）**。它的任务是预测图片中的噪声成分。

$$\text{去噪后图片} = \text{输入噪声图片} - \text{预测噪声}$$

### 3.2 为什么预测噪声比预测干净图片更好？

**原因**：生成一张图片和生成噪声的难度完全不同。

- 如果模型能产生一只**带噪声的猫**，说明它几乎**已经学会了画猫**
- 而**预测噪声**是一个简单得多的任务——只需识别图片中的随机扰动

> **直觉**："判断这张图哪里不对劲"比"从头画一张完美的图"容易得多。扩散模型把"生成"这个困难问题巧妙地转化为"去噪"这个相对简单的问题。

### 3.3 噪声预测器的输入输出

- **输入1**：被噪声污染的图片
- **输入2**：当前去噪步数编号 $t$（如2、100、1000）
- **输出**：该步骤应被去除的噪声图案（与输入图片同形状）

In [ ]:
# ============================================
# PyTorch示例1：噪声预测器的基本结构
# ============================================
import torch
import torch.nn as nn

class NoisePredictor(nn.Module):
    """
    扩散模型的核心组件：噪声预测器
    
    输入：噪声图片 x_t + 时间步 t
    输出：预测的噪声 epsilon_theta (与输入同形状)
    
    关键设计：
    - 时间步t被编码为高维嵌入，注入网络各层
    - 输出与输入同维度（逐像素预测噪声值）
    - 实际实现通常使用U-Net作为骨干
    """
    def __init__(self, in_channels=3, time_emb_dim=256):
        super().__init__()
        # 时间步嵌入：将整数t映射为高维向量
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_emb_dim),
            nn.SiLU(),  # Swish激活函数，扩散模型标配
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        
        # 简化的U-Net骨干
        self.conv1 = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, in_channels, 3, padding=1)
        
        # 时间嵌入投影层
        self.time_proj = nn.Linear(time_emb_dim, 64)
    
    def forward(self, x, t):
        """
        Args:
            x: 噪声图片 [B, C, H, W]
            t: 时间步 [B] 或 [B, 1]
        Returns:
            预测的噪声 [B, C, H, W]
        """
        if t.dim() == 1:
            t = t.unsqueeze(-1).float()
        t_emb = self.time_mlp(t)  # [B, time_emb_dim]
        
        h = self.conv1(x)
        # 将时间信息注入特征图（加法注入）
        t_proj = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + t_proj
        h = torch.relu(h)
        h = self.conv2(h)
        h = torch.relu(h)
        noise_pred = self.conv3(h)
        return noise_pred

print("噪声预测器已定义。")
print("关键设计：时间信息通过加法注入特征图，控制不同阶段的去噪行为。")

## 四、扩散模型的训练

### 4.1 前向过程（扩散过程）——创造训练数据

噪声预测器的训练数据是**人为创造**的。创造方法称为**前向过程（Forward/Diffusion Process）**：

1. 从数据集中取一张真实图片 $x_0$
2. 从高斯分布中随机采样噪声 $\epsilon \sim \mathcal{N}(0, I)$
3. 按比例将噪声加到图片上：$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1-\bar{\alpha}_t} \cdot \epsilon$
4. $t$ 越大，加的噪声越多，图片越"花"
5. $t=T$ 时，$\bar{\alpha}_T \approx 0$，图片几乎就是纯噪声

这就创造了训练数据对：
- **输入**：噪声图片 $x_t$ + 步数 $t$
- **标签**：实际加入的噪声 $\epsilon$

### 4.2 训练目标函数

$$\mathcal{L}_{simple} = \mathbb{E}_{x_0, \epsilon, t}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

各符号含义：
- $x_0$：从数据集中采样的原始干净图片
- $\epsilon \sim \mathcal{N}(0, I)$：随机采样的高斯噪声
- $t \sim \text{Uniform}(1, T)$：随机采样的时间步
- $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$：加噪后的图片
- $\epsilon_\theta$：噪声预测器（参数为 $\theta$）

训练就是让预测噪声 $\epsilon_\theta(x_t, t)$ 尽可能接近实际噪声 $\epsilon$。

### 4.3 训练算法伪代码

```
重复直到收敛：
    1. 取干净图片 x_0
    2. 随机采样时间步 t ~ Uniform(1, T)
    3. 随机采样噪声 epsilon ~ N(0, I)
    4. 计算 x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1-alpha_bar_t)*epsilon
    5. 计算 epsilon_pred = noise_predictor(x_t, t)
    6. 损失 = MSE(epsilon_pred, epsilon)
    7. 梯度下降更新参数
```

In [ ]:
# ============================================
# PyTorch示例2：扩散模型的前向过程和训练
# ============================================
class DiffusionModel:
    """简化的DDPM扩散模型训练框架"""
    
    def __init__(self, noise_predictor, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        """
        Args:
            noise_predictor: 噪声预测器网络
            T: 扩散总步数（默认1000）
            beta_start/end: 噪声调度参数
        """
        self.noise_predictor = noise_predictor
        self.T = T
        self.device = device
        
        # 线性噪声调度
        self.betas = torch.linspace(beta_start, beta_end, T).to(device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)  # 累积乘积
    
    def forward_diffusion(self, x_0, t):
        """
        前向加噪：干净图 -> 噪声图
        x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
        """
        alpha_bar_t = self.alpha_bars[t]
        while alpha_bar_t.dim() < x_0.dim():
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        
        epsilon = torch.randn_like(x_0)  # 采样噪声
        x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * epsilon
        return x_t, epsilon
    
    def train_step(self, x_0, optimizer):
        """单步训练"""
        B = x_0.shape[0]
        t = torch.randint(0, self.T, (B,), device=self.device)
        x_t, epsilon = self.forward_diffusion(x_0, t)
        epsilon_pred = self.noise_predictor(x_t, t)
        loss = nn.functional.mse_loss(epsilon_pred, epsilon)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        return loss.item()

print("扩散模型训练框架已定义。")
print("\n训练关键：")
print("1. 每步随机采样t和noise——数据增强'无限'")
print("2. 损失是MSE——简单稳定")
print("3. 不需要判别器/博弈——训练比GAN容易得多")

## 五、文生图 (Text-to-Image)：把文字条件加入扩散模型

### 5.1 训练数据：LAION

要做文字→图片生成，需要**图文成对数据**。

- ImageNet：每张图片只有类别标签，无文字描述，100万张
- **LAION**：**58.5亿张**图片配文字描述！Midjourney、Stable Diffusion、DALL-E都以此为基础

LAION包含中文、英文等多语言描述——所以这些模型能理解中文，因为它们训练数据里就有中文。

### 5.2 如何加入文字条件

修改方法：在噪声预测器中额外输入文字条件。

- **基础版**：噪声预测器(噪声图片 $x_t$ + 步数 $t$)
- **文生图版**：噪声预测器(噪声图片 $x_t$ + 步数 $t$ + **文字描述**)

文字一般先通过文本编码器（如CLIP的文本编码器）转为向量，再通过**交叉注意力（Cross-Attention）**注入到扩散模型各层中。

### 5.3 训练流程

```
1. 取一张图片 + 它的文字描述
2. 对图片执行前向扩散到某一步t
3. 将（噪声图片，步数t，文字描述）输入噪声预测器
4. 预测噪声并与实际加入的噪声求MSE
5. 梯度下降
```

In [ ]:
# ============================================
# PyTorch示例3：文生图扩散模型框架
# ============================================
class TextConditionedNoisePredictor(nn.Module):
    """
    带文字条件的噪声预测器
    使用交叉注意力让图片特征“关注”文字描述
    """
    def __init__(self, in_channels=3, text_emb_dim=512, time_emb_dim=256):
        super().__init__()
        
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_emb_dim), nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        
        self.conv_in = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.conv_out = nn.Conv2d(64, in_channels, 3, padding=1)
        
        # 交叉注意力：让图片特征attend到文字
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=64, num_heads=4, batch_first=True
        )
        self.text_proj = nn.Linear(text_emb_dim, 64)
        self.time_proj = nn.Linear(time_emb_dim, 64)
    
    def forward(self, x_t, t, text_emb):
        """
        Args:
            x_t: 噪声图片 [B, C, H, W]
            t: 时间步 [B, 1]
            text_emb: 文字嵌入 [B, L, text_emb_dim]
        Returns:
            预测的噪声 [B, C, H, W]
        """
        B, C, H, W = x_t.shape
        t_emb = self.time_mlp(t.float())
        h = self.conv_in(x_t)
        
        # 时间信息注入
        t_feat = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + t_feat
        
        # 交叉注意力：图片特征(Q)关注文字(K,V)
        h_flat = h.view(B, 64, -1).transpose(1, 2)  # [B, H*W, 64]
        text_feat = self.text_proj(text_emb)  # [B, L, 64]
        h_attn, _ = self.cross_attn(h_flat, text_feat, text_feat)
        h = h_attn.transpose(1, 2).view(B, 64, H, W)
        
        noise_pred = self.conv_out(h)
        return noise_pred

print("带文字条件的噪声预测器已定义。")
print("核心机制：交叉注意力(Cross-Attention)")
print("图片特征 -> Query, 文字嵌入 -> Key/Value")
print("让图片生成过程“参考”文字描述。")

## 六、扩散模型 vs GAN：两种生成范式的对比

| 特性 | GAN（第8章） | 扩散模型（本章） |
|------|-------------|-----------------|
| **生成方式** | 一次前向传播直接生成 | 多次迭代去噪（通常1000步） |
| **训练难度** | 极难（博弈平衡微妙） | 简单（明确的MSE回归损失） |
| **生成质量** | 极高（清晰） | 极高，甚至更好 |
| **生成速度** | 快（单步） | 慢（多步迭代） |
| **模式覆盖** | 容易出现模式崩塌/丢失 | 模式覆盖更全面 |
| **数学基础** | 博弈论、JS/Wasserstein散度 | 热力学扩散、变分推断 |
| **代表系统** | StyleGAN, BigGAN | Stable Diffusion, DALL-E, Midjourney |
| **条件生成** | 需要特殊设计的判别器 | 自然支持，通过交叉注意力注入 |

> **核心差异**：GAN像"一步到位"的画家（从噪声直接变图片），扩散模型像"逐步雕刻"的雕塑家。后者的过程更可控、更容易训练，但更慢。两者各有优劣，未来趋势可能是融合。

## 七、核心要点总结

1. **核心思想**：从纯噪声开始，反复去噪最终生成清晰图片——就像雕塑家从石头中"释放"雕塑。

2. **逆过程（生成）**：从 $\mathcal{N}(0,I)$ 采样纯噪声 → 反复应用去噪模块（1000步）→ 清晰图片。步数编号从大到小（1000→1）。

3. **噪声预测器**：去噪模块的核心。输入噪声图片+步数，输出预测的噪声。用"减法"（输入-预测噪声）实现去噪。

4. **为什么预测噪声而非直接输出干净图片**：预测噪声（识别哪里不对）比生成图片（创造对的像素）简单得多。

5. **前向过程**：从真实图片逐步加噪声来创造训练数据。输入=噪声图+步数，标签=加入的噪声。不需要标注数据！

6. **训练损失**：MSE(预测噪声, 真实噪声)——简单稳定的回归损失。这是扩散模型比GAN更容易训练的根本原因。

7. **文生图**：将文字描述编码后通过交叉注意力注入去噪模块。需要图文成对数据（LAION的58.5亿张图片是关键）。

8. **扩散模型优势**：训练稳定（明确损失）、模式覆盖全面（不易崩塌）。代价：推理慢（需多步迭代）。

9. **代表性系统**：DALL-E、Google Imagen、Stable Diffusion、Midjourney都基于扩散模型或其变体。

10. **与GAN的关系**：不是替代而是互补——GAN适合需要实时生成的应用，扩散模型适合追求极致质量的场景。

## 八、练习与思考

1. 用"米开朗基罗雕刻"的类比解释扩散模型的生成过程。两种"创造"有什么相似之处？

2. 为什么去噪模块需要输入步数编号 $t$？如果不用 $t$，单一模块能处理从纯噪声到精细图的全部阶段吗？

3. 论证为什么"预测噪声"比"直接预测干净图片"更容易。从信息量和学习难度的角度分析。

4. 前向过程公式 $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$ 为什么用这个形式？用简单的 $x_t = x_0 + \epsilon \cdot t$ 可以吗？

5. 比较扩散模型和GAN：训练稳定性、生成质量、生成速度、模式多样性——四个维度各有什么优劣？

6. 交叉注意力在文生图扩散模型中起什么作用？如果把交叉注意力替换为简单的拼接会怎样？

7. 实现一个最小化的DDPM，在MNIST上训练。观察不同时间步的噪声图和去噪效果。

8. 为什么MSE损失在扩散模型中效果好，在GAN中就不行？（提示：GAN需要博弈，扩散模型是纯回归）

9. 扩散模型推理为什么慢？有哪些可能的加速方法？

10. LAION数据集有58.5亿张图片——为什么数据量对文生图如此重要？数据量与模型能力之间有什么关系？